# Análisis de Datos Clínicos con NumPy y Matplotlib

> **Módulo:** [3.1 Python — Fundamentos para análisis de datos](https://swcarpentry.github.io/python-novice-inflammation/) | **Fecha:** 11 de septiembre de 2026  
> **Objetivo:** Cargar datos de pacientes en una matriz, calcular estadísticas usando ejes (`axis`), visualizar los resultados en gráficos claros y descubrir por qué algunos datos son falsos o tienen errores de medición.

## 1. Cargar los datos del archivo CSV
Importamos `numpy` y `matplotlib`. Usamos `np.loadtxt` para leer el archivo `inflammation-01.csv`. A diferencia de una lista normal de Python, NumPy guarda los números juntos en la memoria RAM, lo que hace que los cálculos sean ultra rápidos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Carga de datos tabulares delimitados por comas hacia memoria continua (C-order)
data = np.loadtxt(fname='data/inflammation-01.csv', delimiter=',')

# Inspección de propiedades estructurales del arreglo en memoria
print(f"Tipo de estructura: {type(data)}")
print(f"Dimensiones de la matriz (pacientes, días): {data.shape}")
print(f"Tipo de dato de los elementos: {data.dtype}")
print(f"Total de observaciones registradas: {data.size}")

## 2. Rebanar la matriz (Slicing)
Para sacar datos usamos corchetes: `data[filas, columnas]`.
* Recuerda: en Python el primer número se incluye y el último **no se incluye** (`0:3` toma las filas 0, 1 y 2).

In [ ]:
# Acceso a una celda puntual: Paciente 0, Día 0
print(f"Valor inicial (Paciente 0, Día 0): {data[0, 0]}")

# Acceso a una coordenada central: Paciente 30, Día 20
print(f"Valor central (Paciente 30, Día 20): {data[30, 20]}")

# Extracción de una submatriz: primeros 3 pacientes (filas 0 a 2) x primeros 5 días (columnas 0 a 4)
subconjunto = data[0:3, 0:5]
print("\nSubmatriz de muestra (3 pacientes x primeros 5 días):")
print(subconjunto)

## 3. Operaciones usando Ejes (`axis=0` vs `axis=1`)
Aquí usamos el truco de la **prensa hidráulica**:
* **`axis=0` (aplasta hacia abajo):** Aplasta a los 60 pacientes y nos entrega **40 números** (el promedio, máximo o mínimo de cada día).
* **`axis=1` (aplasta hacia los lados):** Aplasta los 40 días y nos entrega **60 números** (el promedio de cada paciente en todo su tratamiento).

In [ ]:
# 1. Agregación a lo largo del eje 0 (comportamiento diario global)
promedio_diario = np.mean(data, axis=0)
maximo_diario = np.max(data, axis=0)
minimo_diario = np.min(data, axis=0)
desviacion_diaria = np.std(data, axis=0)

print(f"Vector de promedios diarios — Longitud: {promedio_diario.shape[0]} días")
print(f"Rango global de inflamación diaria: Mínimo = {minimo_diario.min():.1f} | Máximo = {maximo_diario.max():.1f}")

# 2. Agregación a lo largo del eje 1 (comportamiento individual por paciente)
promedio_pacientes = np.mean(data, axis=1)
print(f"Vector de promedios por paciente — Longitud: {promedio_pacientes.shape[0]} pacientes")

## 4. Mapa de Calor (Ver los 2400 datos juntos)
El mapa de calor pinta cada número como un píxel de color. Los colores claros son dolor alto y los oscuros son dolor bajo. Permite ver de un vistazo cómo avanza la enfermedad en todos los pacientes.

In [ ]:
# Visualización completa de la matriz mediante mapa de calor bidimensional
plt.figure(figsize=(9.0, 4.5))
plt.imshow(data, aspect='auto', cmap='viridis')
plt.colorbar(label='Nivel de Inflamación')
plt.title('Mapa de Calor: Distribución de Inflamación (60 Pacientes x 40 Días)', fontsize=11, weight='bold')
plt.xlabel('Día de Tratamiento (0 a 39)', fontsize=10)
plt.ylabel('Paciente (0 a 59)', fontsize=10)
plt.show()

## 5. Gráficas de Promedio, Máximo y Mínimo (Panel Triple)
Ponemos las 3 curvas una al lado de la otra usando `subplot(1, 3, posición)` para comparar cómo evoluciona la enfermedad día a día.
* Agregamos **grilla mayor y menor** y **marcadores** para poder leer los valores con precisión.

In [ ]:
# Configuración del lienzo general para el panel de 3 gráficos (1 fila x 3 columnas)
fig = plt.figure(figsize=(15.0, 4.8))
fig.suptitle('Evaluación Longitudinal de Inflamación — Archivo 01', x=0.05, y=0.98, ha='left', fontsize=12, weight='bold')

# -------------------------------------------------------------------------
# Subplot 1: Promedio Diario de Inflamación
# -------------------------------------------------------------------------
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(range(40), promedio_diario, linestyle='-', marker='o', markersize=3, color='royalblue', label='Promedio')
ax1.set_title('Promedio Diario', fontsize=11, weight='bold')
ax1.set_xlabel('Día de Tratamiento', fontsize=10)
ax1.set_ylabel('Inflamación Promedio', fontsize=10)
ax1.set_xticks(np.arange(0, 41, 5))
ax1.minorticks_on()
ax1.grid(True, which='major', linestyle='-', alpha=0.6)
ax1.grid(True, which='minor', linestyle=':', alpha=0.3)

# -------------------------------------------------------------------------
# Subplot 2: Máximo Diario (Perfil de Rampa Lineal)
# -------------------------------------------------------------------------
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(range(40), maximo_diario, linestyle='-', marker='s', markersize=3, color='crimson', label='Máximo')
ax2.set_title('Máximo Diario (Rampa Recta)', fontsize=11, weight='bold')
ax2.set_xlabel('Día de Tratamiento', fontsize=10)
ax2.set_ylabel('Inflamación Máxima', fontsize=10)
ax2.set_xticks(np.arange(0, 41, 5))
ax2.set_yticks(np.arange(0, 21, 2))
ax2.minorticks_on()
ax2.grid(True, which='major', linestyle='-', alpha=0.6)
ax2.grid(True, which='minor', linestyle=':', alpha=0.3)

# -------------------------------------------------------------------------
# Subplot 3: Mínimo Diario (Perfil de Escalones Discretos)
# -------------------------------------------------------------------------
ax3 = fig.add_subplot(1, 3, 3)
ax3.step(range(40), minimo_diario, where='mid', color='forestgreen', linewidth=1.5)
ax3.plot(range(40), minimo_diario, linestyle='none', marker='^', markersize=3.5, color='forestgreen', label='Mínimo')
ax3.set_title('Mínimo Diario (Gradas)', fontsize=11, weight='bold')
ax3.set_xlabel('Día de Tratamiento', fontsize=10)
ax3.set_ylabel('Inflamación Mínima', fontsize=10)
ax3.set_xticks(np.arange(0, 41, 4))
ax3.set_yticks(np.arange(0, 6, 1))
ax3.minorticks_on()
ax3.grid(True, which='major', linestyle='-', alpha=0.6)
ax3.grid(True, which='minor', linestyle=':', alpha=0.3)

# Ajuste automático de márgenes para evitar solapamientos
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 6. Funciones para no repetir código
Creamos dos funciones para que el trabajo sea automático:
1. `detectar_anomalias(data)`: Revisa si el archivo tiene la rampa recta o si tiene ceros sospechosos.
2. `analizar(filename)`: Carga el archivo, corre la revisión y dibuja los 3 gráficos.

In [ ]:
def detectar_anomalias(data):
    """
    Audita la matriz de datos para detectar comportamientos matemáticos anómalos.
    
    Parámetros:
        data (np.ndarray): Matriz bidimensional de datos clínicos (pacientes x días).
    """
    max_diario = np.max(data, axis=0)
    min_diario = np.min(data, axis=0)
    
    # Condición 1: Rampa lineal artificial en máximos
    if max_diario[0] == 0 and max_diario[20] == 20:
        print('  ⚠️  ALERTA: Los valores máximos forman una rampa lineal artificial.')
    # Condición 2: Sensores o registros bloqueados en cero absoluto
    elif np.sum(min_diario) == 0:
        print('  ⚠️  ALERTA: Los valores mínimos son cero todos los días (posible falla de sensor).')
    else:
        print('  ✅  COMPORTAMIENTO: Los datos parecen biológicamente naturales.')


def analizar(filename):
    """
    Carga un archivo CSV, ejecuta la detección de anomalías y genera el panel de subplots.
    
    Parámetros:
        filename (str): Ruta al archivo tabular CSV.
    """
    print(f"\n{'=' * 50}")
    print(f"AUDITORÍA CLÍNICA: {filename}")
    print(f"{'=' * 50}")
    
    # Carga de la matriz de datos del archivo correspondiente
    data = np.loadtxt(fname=filename, delimiter=',')
    detectar_anomalias(data)
    
    # Construcción del panel de visualización técnica
    fig = plt.figure(figsize=(15.0, 4.5))
    fig.suptitle(f'Estudio Clínico: {filename}', x=0.05, y=0.98, ha='left', fontsize=12, weight='bold')
    
    # Panel 1: Promedio
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.plot(range(40), np.mean(data, axis=0), linestyle='-', marker='o', markersize=3, color='royalblue')
    ax1.set_title('Promedio Diario', fontsize=10, weight='bold')
    ax1.set_xlabel('Día de Tratamiento', fontsize=9)
    ax1.set_ylabel('Inflamación', fontsize=9)
    ax1.minorticks_on()
    ax1.grid(True, which='major', linestyle='-', alpha=0.5)
    ax1.grid(True, which='minor', linestyle=':', alpha=0.25)
    
    # Panel 2: Máximo
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.plot(range(40), np.max(data, axis=0), linestyle='-', marker='s', markersize=3, color='crimson')
    ax2.set_title('Máximo Diario', fontsize=10, weight='bold')
    ax2.set_xlabel('Día de Tratamiento', fontsize=9)
    ax2.minorticks_on()
    ax2.grid(True, which='major', linestyle='-', alpha=0.5)
    ax2.grid(True, which='minor', linestyle=':', alpha=0.25)
    
    # Panel 3: Mínimo
    ax3 = fig.add_subplot(1, 3, 3)
    ax3.step(range(40), np.min(data, axis=0), where='mid', color='forestgreen')
    ax3.plot(range(40), np.min(data, axis=0), linestyle='none', marker='^', markersize=3.5, color='forestgreen')
    ax3.set_title('Mínimo Diario', fontsize=10, weight='bold')
    ax3.set_xlabel('Día de Tratamiento', fontsize=9)
    ax3.minorticks_on()
    ax3.grid(True, which='major', linestyle='-', alpha=0.5)
    ax3.grid(True, which='minor', linestyle=':', alpha=0.25)
    
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## 7. Procesar varios archivos en bucle (`glob`)
Usamos `glob.glob` para que Python busque solo todos los archivos que se llamen `inflammation-*.csv` y los ordene con `sorted()`. Con un simple bucle `for`, analizamos los 3 primeros archivos en segundos.

In [ ]:
import glob

# Identificación determinista y ordenada de archivos de datos
archivos = sorted(glob.glob('data/inflammation-*.csv'))
print(f"Total de archivos detectados en lote: {len(archivos)}")

# Ejecución del pipeline sobre muestra de control (primeros 3 archivos)
for f in archivos[:3]:
    analizar(f)

## 8. Demostración con derivadas (`np.diff`)
¿Cómo sabemos matemáticamente que la rampa del archivo 01 es falsa y no casualidad?  
Usamos `np.diff()` para restar cada día con el día anterior (la pendiente). Si todos los días de subida la resta da exactamente `1.0`, significa que subió en una línea recta perfecta hecha por una fórmula de computadora.

In [ ]:
# Extracción de máximos del archivo 01
data_01 = np.loadtxt(fname='data/inflammation-01.csv', delimiter=',')
maximos_01 = np.max(data_01, axis=0)

# Cálculo de diferencias consecutivas (derivada discreta) durante los primeros 20 días
diferencias = np.diff(maximos_01[:21])
print("Diferencias consecutivas entre días adyacentes (días 0 al 20):")
print(diferencias)

# Verificación booleana de constancia unitaria
es_recta_perfecta = np.all(diferencias == 1.0)
print(f"\n¿Pendiente estrictamente constante e idéntica a 1.0?: {es_recta_perfecta}")

## 9. Conclusiones sencillas

1. **NumPy vs Listas:** NumPy es mucho más rápido porque guarda los números pegados en la memoria RAM y permite hacer cálculos en todo el arreglo sin escribir bucles `for`.
2. **El truco de los Ejes:** `axis=0` aplasta las filas (pacientes) para dar estadísticas por día. `axis=1` aplasta las columnas (días) para dar estadísticas por paciente.
3. **Gráficas claras:** Poner cuadrículas, marcadores y subplots ayuda a ver patrones que una simple lista de números oculta.
4. **Los datos eran falsos:** Gracias a las gráficas y a `np.diff()`, descubrimos que los archivos 01 y 02 tienen una rampa recta inventada y gradas de 4 días, mientras que el archivo 03 tiene un sensor dañado que registró ceros todos los días.